# Export ONNX

Export the final deployment Keras model to ONNX and write deployment metadata.

In [2]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_07_EXPORT_ONNX_CELL_PROGRESS_1 = start_notebook_cell_progress('07_export_onnx.ipynb', 'Load export dependencies', total_steps=1)

import json
from pathlib import Path

import onnx
import tf2onnx

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())
tf = get_tensorflow()

finish_notebook_cell_progress(NB_07_EXPORT_ONNX_CELL_PROGRESS_1)


[START] 07_export_onnx.ipynb | Load export dependencies [0/1 step] elapsed=0.0s
[START] 00_shared_setup.ipynb | Shared setup bootstrap [0/1 step] elapsed=0.0s
Default processed dataset not ready at C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\data\roboflow_processed_hsv_lab_threshold_roi_224 - run the early notebooks with raw or Excel input, or set overrides.
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [0/1 step] elapsed=0.0s
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [1/1 step] elapsed=0.0s
[RUNNING] 07_export_onnx.ipynb | Load export dependencies | done [0/1 step] elapsed=0.0s
[RUNNING] 07_export_onnx.ipynb | Load export dependencies | done [1/1 step] elapsed=0.0s


In [3]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_07_EXPORT_ONNX_CELL_PROGRESS_2 = start_notebook_cell_progress('07_export_onnx.ipynb', 'Define export helpers', total_steps=1)

def export_model_to_onnx(model_path: Path, onnx_path: Path, opset: int = 13) -> Path:
    model = tf.keras.models.load_model(model_path, compile=False)
    if not hasattr(model, 'output_names'):
        model.output_names = [tensor.name.split(':')[0] for tensor in model.outputs]
    input_shape = tuple(int(dim) if dim is not None else None for dim in model.input_shape)
    ensure_dir(onnx_path.parent)
    tf2onnx.convert.from_keras(
        model,
        input_signature=(tf.TensorSpec(input_shape, tf.float32, name='image_input'),),
        opset=opset,
        output_path=str(onnx_path),
    )
    onnx.checker.check_model(onnx.load(str(onnx_path)))
    return onnx_path


def build_onnx_metadata(model_path: Path, base_metadata: dict[str, object]) -> dict[str, object]:
    model = tf.keras.models.load_model(model_path, compile=False)
    metadata = dict(base_metadata)
    metadata.setdefault('model_name', model.name)
    metadata.setdefault('backbone', 'MobileNetV3Small')
    metadata.setdefault('model_input_mode', 'cnn_only')
    metadata.setdefault('image_crop_mode', metadata.get('input_mode', INPUT_MODE))
    metadata['input_mode'] = metadata.get('input_mode', INPUT_MODE)
    metadata['input_shape'] = list(model.input_shape[1:])
    metadata['labels'] = metadata.get('labels', LABEL_ORDER)
    return metadata

finish_notebook_cell_progress(NB_07_EXPORT_ONNX_CELL_PROGRESS_2)


[START] 07_export_onnx.ipynb | Define export helpers [0/1 step] elapsed=0.0s
[RUNNING] 07_export_onnx.ipynb | Define export helpers | done [0/1 step] elapsed=0.0s
[RUNNING] 07_export_onnx.ipynb | Define export helpers | done [1/1 step] elapsed=0.0s


In [4]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_07_EXPORT_ONNX_CELL_PROGRESS_3 = start_notebook_cell_progress('07_export_onnx.ipynb', 'Export ONNX model', total_steps=1)

MODEL_PATH = Path(str(override('MODEL_PATH', TRAINING_OUTPUTS_ROOT / 'mobilenetv3small_8samples_final_deployment_cnn_only_training1_compatible_end_to_end' / 'models' / 'meatlens_final_8samples_cnn_only_mobilenetv3small.keras')))
ONNX_PATH = Path(str(override('ONNX_PATH', MODEL_PATH.with_suffix('.onnx'))))
ONNX_METADATA_PATH = Path(str(override('ONNX_METADATA_PATH', ONNX_PATH.with_suffix('.metadata.json'))))
BASE_METADATA_PATH = override('BASE_METADATA_PATH', None)
BASE_METADATA = dict(override('BASE_METADATA', {}))
if BASE_METADATA_PATH:
    BASE_METADATA = json.loads(Path(str(BASE_METADATA_PATH)).read_text(encoding='utf-8'))

export_model_to_onnx(MODEL_PATH, ONNX_PATH)
metadata = build_onnx_metadata(MODEL_PATH, BASE_METADATA)
ONNX_METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'onnx_path: {ONNX_PATH}')
print(f'metadata_path: {ONNX_METADATA_PATH}')

finish_notebook_cell_progress(NB_07_EXPORT_ONNX_CELL_PROGRESS_3)


[START] 07_export_onnx.ipynb | Export ONNX model [0/1 step] elapsed=0.0s
onnx_path: C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\training_outputs\roboflow\mobilenetv3small_8samples_final_deployment_cnn_only\models\meatlens_final_8samples_cnn_only_mobilenetv3small.onnx
metadata_path: C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\training_outputs\roboflow\mobilenetv3small_8samples_final_deployment_cnn_only\models\meatlens_final_8samples_cnn_only_mobilenetv3small.metadata.json
[RUNNING] 07_export_onnx.ipynb | Export ONNX model | done [0/1 step] elapsed=35.3s
[RUNNING] 07_export_onnx.ipynb | Export ONNX model | done [1/1 step] elapsed=35.3s
